# Prepare a protein–ligand target

Start from a protein PDB and a ligand SDF to prepare a ligand-containing target surface.

This notebook previews the configuration by default. Edit your paths and set `RUN_TASK = True` when you are ready to run.


## 1. Before you start

- Prepare your PDB and obtain the site and search checkpoints if you want predictions and descriptors.
- Install the `chemistry` extra in this environment.
- Provide the upstream MaSIF source directory and a separate Python interpreter with its original PyMesh and geodesic dependencies. The TensorFlow checkpoint-reader environment alone does not provide these geometry tools.
- Install Reduce, MSMS, PDB2PQR, APBS, and multivalue. These external programs are configured below; see [installation](../README.md#optional-molecular-components).
- Set `pdb2pqr_apbs_input` to `flag` for PDB2PQR 2.1, or `filename` if your version requires `--apbs-input=FILE`.

- Provide a ligand connectivity SDF and identify the ligand residue name and chain in the PDB.

Use the `Python (masif_torch)` kernel. See [installation](../README.md#installation) and the [user guide](../USAGE.md).

For ligand preparation, use the MaSIF-neosurf `masif/source` directory, which includes the ligand preparation utilities.


## 2. Open the project


In [ ]:
from pathlib import Path
import importlib.util
import os
import shlex
import shutil
import subprocess
import sys
import tempfile

from omegaconf import OmegaConf

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch repository or its tutorials directory.")
os.chdir(ROOT)
CLI = [sys.executable, "-m", "masif"]
RUN_TASK = False  # Change to True only after editing the paths below.
print("Repository:", ROOT)
print("Python:", sys.executable)


## 3. Choose your inputs and settings

Paths below are relative to the repository unless you provide an absolute path. Choose a separate output directory for each run.


In [ ]:
TASK = "prepare"
config = OmegaConf.create({
    "task": "preparation", "pdb": "data/input/target.pdb",
    "identifier": "TARGET_A", "output": "results/tutorial_preparation",
    "source_root": "/path/to/masif-neosurf/masif/source",
    "legacy_python": "/path/to/geometry_environment/bin/python",
    "chemistry_python": sys.executable,
    "site_checkpoint": "weights/site/interface_prediction_backup.pt",
    "search_checkpoint": "weights/search/surface_descriptors_historical.pt",
    "precompute": ["site", "ppi_search"], "pdb2pqr_apbs_input": "flag",
    "environment": {
        "MSMS_BIN": "/path/to/msms", "PDB2PQR_BIN": "/path/to/pdb2pqr",
        "APBS_BIN": "/path/to/apbs", "MULTIVALUE_BIN": "/path/to/multivalue",
        "REDUCE_HET_DICT": "/path/to/reduce_wwPDB_het_dict.txt",
        "PATH": "/path/to/reduce_directory" + os.pathsep + os.environ.get("PATH", ""),
    },
})
# Use a PDB identifier plus protein chain, for example 1A7X_A.
# Set a checkpoint to None if you only need its precomputed surface data.

config.pdb = "data/input/protein_ligand.pdb"
config.output = "results/tutorial_ligand_preparation"
config.ligand = "FKA_B"  # Ligand residue name and chain in the PDB.
config.sdf = "data/input/ligand.sdf"
config.source_root = "/path/to/masif-neosurf/masif/source"


## 4. Preview your configuration

This prints the Hydra configuration without loading your input data or running the task.


In [ ]:
config_directory = tempfile.TemporaryDirectory(prefix="masif_tutorial_")
config_path = Path(config_directory.name) / "task.yaml"
OmegaConf.save(config, config_path)
command = CLI + [TASK, "--config", str(config_path)]
print(shlex.join(command))
subprocess.run(command + ["--cfg", "job"], check=True)


## 5. Run when ready

After editing the settings, set `RUN_TASK = True` in the setup cell and rerun the notebook. The checks below help identify missing inputs before starting.


In [ ]:
if RUN_TASK:
    for key in ("pdb", "sdf", "site_checkpoint", "search_checkpoint"):
        filename = config.get(key)
        if filename and not Path(filename).is_file():
            raise FileNotFoundError(f"Missing {key}: {filename}. Update the settings before running.")
    if not Path(config.source_root, "data_preparation/04-masif_precompute.py").is_file():
        raise FileNotFoundError("Set source_root to the neosurf masif/source directory containing data_preparation.")
    if not Path(config.source_root, "triangulation/ligand_utils.py").is_file():
        raise FileNotFoundError("Use the neosurf masif/source directory containing triangulation/ligand_utils.py.")
    for key in ("legacy_python", "chemistry_python"):
        if not Path(config[key]).is_file() or not os.access(config[key], os.X_OK):
            raise FileNotFoundError(f"Set {key} to an executable Python interpreter.")
    for key in ("MSMS_BIN", "PDB2PQR_BIN", "APBS_BIN", "MULTIVALUE_BIN"):
        if not Path(config.environment[key]).is_file() or not os.access(config.environment[key], os.X_OK):
            raise FileNotFoundError(f"Set environment.{key} to the installed executable.")
    if shutil.which("reduce", path=config.environment.PATH) is None:
        raise FileNotFoundError("Add the directory containing reduce to environment.PATH.")
    if not Path(config.environment.REDUCE_HET_DICT).is_file():
        raise FileNotFoundError("Set REDUCE_HET_DICT to the Reduce heterogen dictionary.")
    subprocess.run([config.chemistry_python, "-c", "import rdkit, Bio"], check=True)
    subprocess.run([config.legacy_python, "-c", "import pymesh, sklearn"], check=True)
    subprocess.run(command, check=True)
else:
    print("Preview only. Edit your paths and set RUN_TASK = True to run.")


## 6. Next steps

Pack the processed output as a target in [seed docking](seed.ipynb), or score positioned binders with [MaSIF-neosurface](neosurface.ipynb). Preserve the target and ligand coordinate system.
